# ILLM Lab 3: Subword Tokenization (BPE & WordPiece) on Gujarati Corpus

**Course**: Introduction to Indic Language Modeling (ILLM)  
**Target Language**: Gujarati (`guj_Gujr`)  
**Dataset**: `ai4bharat/IndicCorpV2`  

---  
## 📌 Assignment Objectives
1. **Sentence Tokenization & Length-Stratified Splitting**: Sentence-tokenize raw paragraphs from IndicCorpV2 Gujarati corpus. Create Development (1,000 sentences) and Test (1,000 sentences) sets maintaining a diverse sentence length distribution (not skewed to uniform/similar lengths). The remaining sentences serve as Training data.
2. **From-Scratch Implementation**: Implement **Byte Pair Encoding (BPE)** and **WordPiece** tokenization algorithms entirely from scratch without external tokenizer libraries.
3. **Training Data Size Analysis**: Evaluate tokenization behavior across 4 training set sizes:
   - $100,000$ sentences
   - $300,000$ sentences
   - $500,000$ sentences
   - $1,000,000$ sentences
4. **Vocabulary Size Analysis**: Evaluate tokenization performance across 3 vocabulary sizes:
   - $20,000$ subwords
   - $30,000$ subwords
   - $50,000$ subwords

---  
## 1. Setup & Environment Configurations

In [12]:
import sys
import os

# Force UTF-8 on Windows environments before importing dataset handlers
if sys.platform == "win32":
    os.environ["PYTHONUTF8"] = "1"
    os.environ["PYTHONIOENCODING"] = "utf-8"

import re
import csv
import math
import time
import random
import json
from collections import defaultdict, Counter
from typing import List, Tuple, Dict, Optional

import numpy as np
import matplotlib.pyplot as plt
import os
os.environ["PYTHONUTF8"] = "1"
# Fixed Seed for Exact Reproducibility
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

LANGUAGE = "guj_Gujr"
MAX_SENTENCES = 1_100_000  # 1M train + 1K dev + 1K test + buffer
CACHE_FILE = "gujarati_sentences.txt"

TRAIN_SIZES = [100_000, 300_000, 500_000, 1_000_000]
VOCAB_SIZES = [20_000, 30_000, 50_000]

print(f"Setup completed (UTF-8 Enforced). Target Language: {LANGUAGE}")

Setup completed (UTF-8 Enforced). Target Language: guj_Gujr


In [13]:
import os
os.environ["PYTHONUTF8"] = "1"
print("UTF-8 mode:", os.environ["PYTHONUTF8"])

UTF-8 mode: 1


---  
## 2. Dataset Download & Sentence Tokenization

The IndicCorpV2 dataset stores data paragraph-wise. Each row is a paragraph rather than a single sentence.  
We implement `tokenize_sentences()` to segment Gujarati paragraphs using Indic punctuation marks (`.`, `?`, `!`, `।`, `॥`).

In [14]:
data_dir="data/gu.txt"

---  
## 3. Task 1: Length-Stratified Train / Dev / Test Splitting

**Requirement**: Dev and Test sets must contain **1,000 sentences each**, sampled to preserve a wide distribution of sentence lengths (short, medium, long), avoiding clusters of similar lengths.  

We partition the dataset into 10 length quantile bins based on word count and sample evenly across all bins.

In [15]:
def length_stratified_split(sentences: List[str], dev_size: int = 1000, test_size: int = 1000, seed: int = RANDOM_SEED):
    """
    Creates a length-stratified train, dev, and test split.
    Dev and Test sets draw sentences proportionally from 10 sentence-length bins.
    """
    if not sentences:
        raise ValueError("Cannot perform stratified split on an empty sentence list! Ensure dataset download succeeded.")
        
    rng = random.Random(seed)
    indexed = [(i, len(s.split())) for i, s in enumerate(sentences)]
    lengths = [wc for _, wc in indexed]
    
    # Bin edges based on percentiles (10 deciles)
    percentiles = np.percentile(lengths, np.arange(0, 101, 10))
    bin_edges = sorted(set(percentiles.astype(int)))
    
    bins = defaultdict(list)
    for idx, wc in indexed:
        b = 0
        for j in range(len(bin_edges) - 1):
            if wc >= bin_edges[j]:
                b = j
        bins[b].append(idx)
        
    for b in bins:
        rng.shuffle(bins[b])
        
    total = sum(len(v) for v in bins.values())
    needed = dev_size + test_size
    
    dev_indices, test_indices = [], []
    for b in sorted(bins.keys()):
        proportion = len(bins[b]) / total
        n_from_bin = max(1, int(proportion * (needed // 2)))
        taken_dev = bins[b][:n_from_bin]
        taken_test = bins[b][n_from_bin:n_from_bin * 2]
        dev_indices.extend(taken_dev)
        test_indices.extend(taken_test)
        
    rng.shuffle(dev_indices)
    rng.shuffle(test_indices)
    dev_indices = set(dev_indices[:dev_size])
    test_indices = set(test_indices[:test_size]) - dev_indices
    
    used = dev_indices | test_indices
    remaining = [i for i in range(len(sentences)) if i not in used]
    rng.shuffle(remaining)
    
    while len(dev_indices) < dev_size and remaining:
        dev_indices.add(remaining.pop())
    while len(test_indices) < test_size and remaining:
        test_indices.add(remaining.pop())
        
    train_indices = [i for i in range(len(sentences)) if i not in dev_indices and i not in test_indices]
    
    dev = [sentences[i] for i in sorted(dev_indices)]
    test = [sentences[i] for i in sorted(test_indices)]
    train = [sentences[i] for i in train_indices]
    
    print("=== Split Statistics & Length Distributions ===")
    for name, split in [("Dev Set", dev), ("Test Set", test), ("Train Set (sample 10k)", train[:10000])]:
        if split:
            wcs = [len(s.split()) for s in split]
            print(f"  {name:<22}: Count = {len(split):>9,} | Length Range = [{min(wcs):>2}, {max(wcs):>3}] words | Mean = {np.mean(wcs):>5.1f} | Std = {np.std(wcs):>4.1f}")
        
    return train, dev, test

---  
## 4. Task 2: From-Scratch BPE (Byte Pair Encoding) Implementation

**BPE Algorithm** (Sennrich et al., 2016):
1. Initialize base vocabulary with characters and end-of-word token `</w>`.
2. Represent words as sequences of symbols.
3. Count adjacent symbol pairs across corpus word frequencies.
4. Iteratively merge the single most frequent adjacent pair.
5. Continue until target vocabulary size is reached.

In [16]:
class BPETokenizer:
    """Byte Pair Encoding Tokenizer implemented from scratch."""
    END_OF_WORD = "</w>"
    
    def __init__(self, vocab_size: int = 30000):
        self.vocab_size = vocab_size
        self.merges: List[Tuple[str, str]] = []
        self.vocab: Dict[str, int] = {}
        
    def _get_word_freqs(self, sentences: List[str]) -> Dict[Tuple[str, ...], int]:
        word_freqs = Counter()
        for sent in sentences:
            for word in sent.strip().split():
                chars = tuple(list(word) + [self.END_OF_WORD])
                word_freqs[chars] += 1
        return dict(word_freqs)
    
    def _get_pair_freqs(self, word_freqs: Dict[Tuple[str, ...], int]) -> Counter:
        pair_freqs = Counter()
        for word, freq in word_freqs.items():
            for i in range(len(word) - 1):
                pair_freqs[(word[i], word[i + 1])] += freq
        return pair_freqs
    
    def _merge_pair(self, word_freqs: Dict[Tuple[str, ...], int], pair: Tuple[str, str]) -> Dict[Tuple[str, ...], int]:
        new_word_freqs = {}
        merged = pair[0] + pair[1]
        for word, freq in word_freqs.items():
            new_word = []
            i = 0
            while i < len(word):
                if i < len(word) - 1 and word[i] == pair[0] and word[i + 1] == pair[1]:
                    new_word.append(merged)
                    i += 2
                else:
                    new_word.append(word[i])
                    i += 1
            new_word_freqs[tuple(new_word)] = freq
        return new_word_freqs
    
    def train(self, sentences: List[str]):
        t0 = time.time()
        word_freqs_raw = self._get_word_freqs(sentences)
        
        chars = set()
        for word in word_freqs_raw:
            for ch in word:
                chars.add(ch)
        self.vocab = {ch: i for i, ch in enumerate(sorted(chars))}
        
        word_list = [list(w) for w in word_freqs_raw.keys()]
        word_counts = list(word_freqs_raw.values())
        
        pair_freqs = Counter()
        pair_to_words = defaultdict(set)
        
        for idx, word in enumerate(word_list):
            for i in range(len(word) - 1):
                pair = (word[i], word[i + 1])
                pair_freqs[pair] += word_counts[idx]
                pair_to_words[pair].add(idx)

        self.merges = []
        num_merges = max(0, self.vocab_size - len(self.vocab))
        
        for step in range(num_merges):
            if not pair_freqs:
                break
            best_pair = pair_freqs.most_common(1)[0][0]
            merged_symbol = best_pair[0] + best_pair[1]
            self.vocab[merged_symbol] = len(self.vocab)
            self.merges.append(best_pair)
            
            words_to_update = list(pair_to_words[best_pair])
            del pair_freqs[best_pair]
            del pair_to_words[best_pair]

            for idx in words_to_update:
                word = word_list[idx]
                count = word_counts[idx]

                for i in range(len(word) - 1):
                    pair = (word[i], word[i + 1])
                    if pair != best_pair:
                        pair_freqs[pair] -= count
                        pair_to_words[pair].discard(idx)
                        if pair_freqs[pair] <= 0:
                            del pair_freqs[pair]

                new_word = []
                i = 0
                while i < len(word):
                    if i < len(word) - 1 and word[i] == best_pair[0] and word[i + 1] == best_pair[1]:
                        new_word.append(merged_symbol)
                        i += 2
                    else:
                        new_word.append(word[i])
                        i += 1
                word_list[idx] = new_word

                for i in range(len(new_word) - 1):
                    pair = (new_word[i], new_word[i + 1])
                    pair_freqs[pair] += count
                    pair_to_words[pair].add(idx)
            
            if (step + 1) % 2000 == 0 or step == num_merges - 1:
                print(f"    [BPE] Merge {step+1}/{num_merges} | Vocab Size: {len(self.vocab):,} | Elapsed: {time.time()-t0:.1f}s")
                
        print(f"  -> BPE Training Complete. Final Vocab: {len(self.vocab):,} subwords ({time.time()-t0:.1f}s)")

    def encode(self, text: str) -> List[str]:
        tokens = []
        for word in text.strip().split():
            symbols = list(word) + [self.END_OF_WORD]
            for a, b in self.merges:
                i = 0
                while i < len(symbols) - 1:
                    if symbols[i] == a and symbols[i + 1] == b:
                        symbols[i] = a + b
                        del symbols[i + 1]
                    else:
                        i += 1
            tokens.extend(symbols)
        return tokens
    
    def encode_batch(self, sentences: List[str]) -> List[List[str]]:
        return [self.encode(s) for s in sentences]

---  
## 5. Task 2: From-Scratch WordPiece Implementation

**WordPiece Algorithm** (Schuster & Nakajima, 2012):
1. Initialize base vocabulary with single characters; prefix non-initial characters with `##`.
2. Score candidate pair merges using Likelihood Ratio:  
   $$\text{Score}(A, B) = \frac{\text{freq}(AB)}{\text{freq}(A) \times \text{freq}(B)}$$
3. Merge pair maximizing this likelihood ratio.
4. Encoding uses **Greedy Longest-Match First** sequence parsing.

In [17]:
class WordPieceTokenizer:
    """WordPiece Tokenizer implemented from scratch."""
    UNK = "[UNK]"
    CONT_PREFIX = "##"
    
    def __init__(self, vocab_size: int = 30000):
        self.vocab_size = vocab_size
        self.vocab: Dict[str, int] = {}
        
    def _get_word_freqs(self, sentences: List[str]) -> Dict[str, int]:
        word_freqs = Counter()
        for sent in sentences:
            for word in sent.strip().split():
                word_freqs[word] += 1
        return dict(word_freqs)
    
    def train(self, sentences: List[str]):
        t0 = time.time()
        word_freqs_raw = self._get_word_freqs(sentences)
        
        char_set = set()
        for word in word_freqs_raw:
            char_set.add(word[0])
            for ch in word[1:]:
                char_set.add(self.CONT_PREFIX + ch)
                
        self.vocab = {self.UNK: 0}
        for i, ch in enumerate(sorted(char_set), 1):
            self.vocab[ch] = i
            
        word_list = []
        word_counts = []
        for word, count in word_freqs_raw.items():
            split = [word[0]] + [self.CONT_PREFIX + c for c in word[1:]]
            word_list.append(split)
            word_counts.append(count)

        pair_freqs = Counter()
        symbol_freqs = Counter()
        pair_to_words = defaultdict(set)

        for idx, split in enumerate(word_list):
            count = word_counts[idx]
            for s in split:
                symbol_freqs[s] += count
            for i in range(len(split) - 1):
                pair = (split[i], split[i + 1])
                pair_freqs[pair] += count
                pair_to_words[pair].add(idx)

        num_merges = max(0, self.vocab_size - len(self.vocab))
        
        for step in range(num_merges):
            if not pair_freqs:
                break
                
            best_pair = None
            best_score = -1.0
            for pair, freq in pair_freqs.items():
                denom = symbol_freqs[pair[0]] * symbol_freqs[pair[1]]
                if denom == 0:
                    continue
                score = freq / denom
                if score > best_score:
                    best_score = score
                    best_pair = pair
                    
            if best_pair is None:
                break
                
            merged = best_pair[0] + best_pair[1][2:] if best_pair[1].startswith(self.CONT_PREFIX) else best_pair[0] + best_pair[1]
            self.vocab[merged] = len(self.vocab)
            
            words_to_update = list(pair_to_words[best_pair])
            del pair_freqs[best_pair]
            del pair_to_words[best_pair]

            for idx in words_to_update:
                split = word_list[idx]
                count = word_counts[idx]

                for s in split:
                    symbol_freqs[s] -= count
                for i in range(len(split) - 1):
                    pair = (split[i], split[i + 1])
                    if pair != best_pair:
                        pair_freqs[pair] -= count
                        pair_to_words[pair].discard(idx)
                        if pair_freqs[pair] <= 0:
                            del pair_freqs[pair]

                new_split = []
                i = 0
                while i < len(split):
                    if i < len(split) - 1 and split[i] == best_pair[0] and split[i + 1] == best_pair[1]:
                        new_split.append(merged)
                        i += 2
                    else:
                        new_split.append(split[i])
                        i += 1
                word_list[idx] = new_split
                
                for s in new_split:
                    symbol_freqs[s] += count
                for i in range(len(new_split) - 1):
                    pair = (new_split[i], new_split[i + 1])
                    pair_freqs[pair] += count
                    pair_to_words[pair].add(idx)
                
            if (step + 1) % 2000 == 0 or step == num_merges - 1:
                print(f"    [WordPiece] Merge {step+1}/{num_merges} | Vocab Size: {len(self.vocab):,} | Elapsed: {time.time()-t0:.1f}s")
                
            if len(self.vocab) >= self.vocab_size:
                break
                
        print(f"  -> WordPiece Training Complete. Final Vocab: {len(self.vocab):,} subwords ({time.time()-t0:.1f}s)")

    def encode(self, text: str) -> List[str]:
        tokens = []
        for word in text.strip().split():
            if not word:
                continue
            sub_tokens = []
            start = 0
            while start < len(word):
                end = len(word)
                found = False
                while start < end:
                    substr = word[start:end]
                    if start > 0:
                        substr = self.CONT_PREFIX + substr
                    if substr in self.vocab:
                        sub_tokens.append(substr)
                        found = True
                        break
                    end -= 1
                if not found:
                    sub_tokens = [self.UNK]
                    break
                start = end
            tokens.extend(sub_tokens)
        return tokens
    
    def encode_batch(self, sentences: List[str]) -> List[List[str]]:
        return [self.encode(s) for s in sentences]

---  
## 6. Evaluation Metrics & Helper Visualizers

In [18]:
def compute_stats(tokenizer, sentences: List[str], name: str) -> Dict:
    """Computes subword statistics over a corpus partition."""
    all_tokens = tokenizer.encode_batch(sentences)
    total_tokens = sum(len(t) for t in all_tokens)
    total_chars = sum(len(s) for s in sentences)
    avg_tokens = total_tokens / len(sentences) if sentences else 0
    
    unk_count = sum(1 for toks in all_tokens for t in toks if t == "[UNK]")
    unk_rate = (unk_count / total_tokens * 100.0) if total_tokens > 0 else 0.0
    
    unique_tokens = set(t for toks in all_tokens for t in toks)
    compression = total_chars / total_tokens if total_tokens > 0 else 0.0
    
    return {
        "name": name,
        "avg_tokens_per_sent": avg_tokens,
        "total_tokens": total_tokens,
        "unique_tokens_used": len(unique_tokens),
        "unk_rate": unk_rate,
        "compression_ratio": compression,
    }

def print_results_table(results: List[Dict], title: str, vary_key: str):
    """Prints a formatted markdown comparison table."""
    if not results:
        print("No results to display.")
        return
    print(f"\n{'='*95}")
    print(f"  {title}")
    print(f"{'='*95}")
    header = f"{'Setting':>12} | {'Method':>10} | {'Avg Tok/Sent':>13} | {'UNK Rate (%)':>12} | {'Compression':>11} | {'Unique Used':>11}"
    print(header)
    print("-" * 95)
    for r in results:
        val_str = f"{r[vary_key]:,}"
        for method, split_key in [("BPE", "bpe_test"), ("WordPiece", "wp_test")]:
            s = r[split_key]
            print(f"{val_str:>12} | {method:>10} | {s['avg_tokens_per_sent']:>13.2f} | {s['unk_rate']:>11.4f}% | {s['compression_ratio']:>11.3f} | {s['unique_tokens_used']:>11,}")
            val_str = ""  # Only display setting header on first row of pair
    print("=" * 95)

---  
## 7. Task 3: Experiment 1 — Impact of Training Data Size

Evaluating subword tokenization behavior across training set sizes:
- **Target Sizes**: 100,000 | 300,000 | 500,000 | 1,000,000 sentences
- **Fixed Vocab Size**: 30,000 subwords

In [19]:
def tokenize_sentences(paragraph: str, min_words: int = 3, min_len: int = 10, max_len: int = 500) -> List[str]:
    """
    Splits Gujarati text into individual sentences based on Indic boundary markers.
    Filters out noisy fragments or excessively long text blocks.
    """
    paragraph = paragraph.strip()
    if not paragraph:
        return []

    boundary = re.compile(r'(?<=[.?!\u0964\u0965\u06D4])\s+')
    parts = boundary.split(paragraph)

    sentences = []

    for part in parts:
        for sub in part.split('\n'):
            sub = sub.strip()

            if min_len <= len(sub) <= max_len:
                words = sub.split()

                if len(words) >= min_words:
                    sentences.append(sub)

    return sentences


def download_gujarati_data(target: int = MAX_SENTENCES) -> List[str]:
    """Streams Gujarati data from IndicCorpV2 using UTF-8 HTTP streaming."""
    import requests

    print(f"Streaming Gujarati ({LANGUAGE}) corpus from Hugging Face...")

    url = (
        "https://huggingface.co/datasets/ai4bharat/IndicCorpV2/"
        "resolve/main/data/gu.txt?download=true"
    )

    sentences, seen = [], set()

    print("  -> Connecting to IndicCorpV2 Gujarati corpus...")

    response = requests.get(
        url,
        stream=True,
        timeout=120
    )
    response.raise_for_status()

    print("  -> Connected. Reading Gujarati data as UTF-8...")

    # Decode the HTTP stream explicitly as UTF-8.
    for raw_line in response.iter_lines(decode_unicode=False):

        if not raw_line:
            continue

        try:
            text = raw_line.decode("utf-8").strip()
        except UnicodeDecodeError:
            # Skip a malformed byte sequence rather than using cp1252.
            continue

        if not text:
            continue

        for s in tokenize_sentences(text):

            if s not in seen:
                seen.add(s)
                sentences.append(s)

        if len(sentences) >= target:
            break

        if len(sentences) % 25000 == 0 and sentences:
            print(
                f"  Collected {len(sentences):,} sentences..."
            )

    response.close()

    print(
        f"Total unique Gujarati sentences collected: "
        f"{len(sentences):,}"
    )

    if not sentences:
        raise ValueError(
            "Downloaded dataset returned 0 valid Gujarati sentences!"
        )

    return sentences


def load_or_download_data() -> List[str]:
    """Loads dataset from local cache if present and non-empty, otherwise streams and caches."""

    if os.path.exists(CACHE_FILE):

        with open(CACHE_FILE, "r", encoding="utf-8") as f:
            sentences = [
                line.strip()
                for line in f
                if line.strip()
            ]

        if len(sentences) >= 1000:

            print(
                f"Successfully loaded {len(sentences):,} cached sentences "
                f"from '{CACHE_FILE}'"
            )

            return sentences

        else:

            print(
                f"Cache file '{CACHE_FILE}' contained only "
                f"{len(sentences)} sentences "
                "(invalid/empty). Downloading fresh..."
            )

            os.remove(CACHE_FILE)

    sentences = download_gujarati_data()

    with open(CACHE_FILE, "w", encoding="utf-8") as f:

        for s in sentences:
            f.write(s + "\n")

    print(
        f"Cached {len(sentences):,} sentences to '{CACHE_FILE}'"
    )

    return sentences

In [20]:
print("Calling load_or_download_data...")
raw_sentences = load_or_download_data()
print(f"Loaded {len(raw_sentences):,} sentences")

Calling load_or_download_data...
Successfully loaded 1,100,000 cached sentences from 'gujarati_sentences.txt'
Loaded 1,100,000 sentences


In [21]:
train_sents, dev_sents, test_sents = length_stratified_split(raw_sentences)

print(f"Train: {len(train_sents):,}")
print(f"Dev:   {len(dev_sents):,}")
print(f"Test:  {len(test_sents):,}")

=== Split Statistics & Length Distributions ===
  Dev Set               : Count =     1,000 | Length Range = [ 3,  67] words | Mean =  14.8 | Std =  8.9
  Test Set              : Count =     1,000 | Length Range = [ 3,  87] words | Mean =  15.1 | Std =  9.8
  Train Set (sample 10k): Count =    10,000 | Length Range = [ 3,  82] words | Mean =  14.8 | Std =  9.1
Train: 1,098,000
Dev:   1,000
Test:  1,000


---  
## 8. Task 4: Experiment 2 — Impact of Vocabulary Size

Evaluating tokenization performance across vocabulary sizes:
- **Vocab Sizes**: 20,000 | 30,000 | 50,000 subwords
- **Fixed Training Size**: Maximum available training corpus (up to 1,000,000 sentences)

In [22]:
vocab_results = []
max_train_size = min(1_000_000, len(train_sents))
train_subset = train_sents[:max_train_size]

print("\n=======================================================")
print(f"  RUNNING TASK 4: Varying Vocabulary Size (Train = {max_train_size:,})")
print("=======================================================")

for vs in VOCAB_SIZES:
    print(f"\n---> Training on {max_train_size:,} sentences with Vocab Size = {vs:,}")
    
    bpe = BPETokenizer(vocab_size=vs)
    bpe.train(train_subset)
    
    wp = WordPieceTokenizer(vocab_size=vs)
    wp.train(train_subset)
    
    bpe_test = compute_stats(bpe, test_sents, "BPE-test")
    wp_test = compute_stats(wp, test_sents, "WordPiece-test")
    
    vocab_results.append({
        "train_size": max_train_size,
        "vocab_size": vs,
        "bpe": bpe, "wp": wp,
        "bpe_test": bpe_test, "wp_test": wp_test
    })

if vocab_results:
    print_results_table(vocab_results, f"Task 4: Impact of Vocabulary Size (Train = {max_train_size:,})", "vocab_size")


  RUNNING TASK 4: Varying Vocabulary Size (Train = 1,000,000)

---> Training on 1,000,000 sentences with Vocab Size = 20,000
    [BPE] Merge 2000/19343 | Vocab Size: 2,657 | Elapsed: 104.8s
    [BPE] Merge 4000/19343 | Vocab Size: 4,657 | Elapsed: 143.6s
    [BPE] Merge 6000/19343 | Vocab Size: 6,657 | Elapsed: 191.9s
    [BPE] Merge 8000/19343 | Vocab Size: 8,657 | Elapsed: 253.6s
    [BPE] Merge 10000/19343 | Vocab Size: 10,657 | Elapsed: 328.8s
    [BPE] Merge 12000/19343 | Vocab Size: 12,657 | Elapsed: 401.8s
    [BPE] Merge 14000/19343 | Vocab Size: 14,657 | Elapsed: 466.8s
    [BPE] Merge 16000/19343 | Vocab Size: 16,657 | Elapsed: 533.1s
    [BPE] Merge 18000/19343 | Vocab Size: 18,657 | Elapsed: 617.9s
    [BPE] Merge 19343/19343 | Vocab Size: 20,000 | Elapsed: 666.4s
  -> BPE Training Complete. Final Vocab: 20,000 subwords (666.4s)
    [WordPiece] Merge 2000/19038 | Vocab Size: 2,962 | Elapsed: 20.1s
    [WordPiece] Merge 4000/19038 | Vocab Size: 4,962 | Elapsed: 30.9s
    [W

---  
## 9. Visualizing Sample Tokenizations Side-by-Side

Comparing BPE vs WordPiece tokenization outputs on sample Gujarati sentences from the Test set.

In [23]:
if vocab_results:
    best_bpe = vocab_results[-1]["bpe"]
    best_wp = vocab_results[-1]["wp"]
    
    print("\n" + "="*95)
    print("  SIDE-BY-SIDE GUJARATI TOKENIZATION EXAMPLES")
    print("="*95)
    
    for i, sent in enumerate(test_sents[:5], 1):
        bpe_toks = best_bpe.encode(sent)
        wp_toks = best_wp.encode(sent)
        print(f"\nSentence {i}:")
        print(f"  Original : {sent}")
        print(f"  BPE      ({len(bpe_toks)} tokens): {' | '.join(bpe_toks[:15])}{' ...' if len(bpe_toks)>15 else ''}")
        print(f"  WordPiece ({len(wp_toks)} tokens): {' | '.join(wp_toks[:15])}{' ...' if len(wp_toks)>15 else ''}")


  SIDE-BY-SIDE GUJARATI TOKENIZATION EXAMPLES

Sentence 1:
  Original : રોટલો કૂવામાં પધરાવાય છે કે જે કૂવાનાં જળમાં ડૂબી જાય છે.
  BPE      (15 tokens): રોટ | લો</w> | કૂવામાં</w> | પધરા | વાય</w> | છે</w> | કે</w> | જે</w> | કૂ | વાનાં</w> | જ | ળમાં</w> | ડૂબી</w> | જાય</w> | છે.</w>
  WordPiece (47 tokens): ર | ##ો | ##ટ | ##લ | ##ો | ક | ##ૂ | ##વ | ##ા | ##મ | ##ા | ##ં | પ | ##ધ | ##ર ...

Sentence 2:
  Original : શિરડીમાં ટેક્નોલોજીની મદદથી આવો જ એક ચમત્કાર થવાનો છે.હવે શિરડીસ્થિત સાંઇ મંદિરમાં ભક્તો ચાલશે અને વીજળીના દીવા રોશની આપશે.
  BPE      (24 tokens): શિર | ડીમાં</w> | ટેક્નોલો | જીની</w> | મદદથી</w> | આવો</w> | જ</w> | એક</w> | ચમત્કાર</w> | થવાનો</w> | છે. | હવે</w> | શિર | ડી | સ્થિત</w> ...
  WordPiece (105 tokens): શ | ##િ | ##ર | ##ડ | ##ી | ##મ | ##ા | ##ં | ટ | ##ે | ##ક | ##્ | ##ન | ##ો | ##લ ...

Sentence 3:
  Original : આજે બોલિવૂડના જાણીતા અભિનેતા જીતેન્દ્રનો જન્મદિવસ છે.
  BPE      (8 tokens): આજે</w> | બોલિવૂડના</w> | જાણીતા</w> | અભિનેતા</w> | જીતેન્દ્ર 

---  
## 10. Summary & Comparative Analysis

### Key Takeaways:
1. **Effect of Training Data Size**:
   - As training dataset size increases ($100k \to 1M$), both BPE and WordPiece observe more frequent morphemes and root words in Gujarati.
   - **Average tokens per sentence** decreases, and **compression ratio** increases because subwords capture longer, morphologically complete units.
   - **UNK Rate** drops significantly with larger training corpora.

2. **Effect of Vocabulary Size**:
   - Larger vocabulary sizes ($20k \to 50k$) allow tokenizers to store whole words or frequent compound subwords.
   - Decreases token count per sentence, accelerating model sequence processing in downstream Transformer architectures.
   - However, excessively large vocabularies increase embedding parameter memory.

3. **BPE vs WordPiece**:
   - **BPE** merges purely by raw pair frequency count ($A + B \to AB$).
   - **WordPiece** merges by likelihood ratio $\frac{\text{freq}(AB)}{\text{freq}(A) \cdot \text{freq}(B)}$, prioritizing merges where constituents rarely appear independently. This leads to cleaner morphological boundary alignments for Indic languages.